[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/04_Finetuning_LowCompute/04_finetune_clip_custom_data/04_finetune_clip_custom_data.ipynb)

# 04. Finetune CLIP on Your Own Custom Data

**This is the practical capstone of Module 04.**

**This notebook covers:**
- Prepare your own image-text dataset
- Finetune OpenCLIP with LoRA (low compute)
- Evaluate: zero-shot classification on your domain
- Visualize before vs after finetuning
- Export and save your adapter

---

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/04_Finetuning_LowCompute/04_finetune_clip_custom_data")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import time
from utils.visualization import *
from utils.helpers import *

set_style()
device = get_device()

## 1. Prepare Your Dataset

CLIP finetuning needs **image-text pairs**. Options:

| Source | Format | Low-Compute? |
|--------|--------|---------|
| Your own images + captions | folder of images + CSV | Yes |
| HuggingFace datasets | `datasets.load_dataset()` | Yes |
| COCO Captions | 330K image-text pairs | Medium |
| Flickr30k | 30K image-text pairs | Yes |

We'll use **synthetic data** here so no downloads are needed.

### Data Quality Guidelines

| Factor | Good | Bad | Impact |
|--------|------|-----|--------|
| Caption length | 10–30 words | 1–3 words | Short captions = weak supervision |
| Diversity | Varied scenes, angles | Same background | Overfitting risk |
| Negatives | Distinct concepts | Near-duplicates | Easy negatives = no learning |
| Size | 1K–100K pairs | <100 | Underfitting |
| Quality | Accurate descriptions | Generic/wrong | Noise in gradients |

In [ ]:
# Create a richer synthetic dataset
images, texts, labels = create_synthetic_image_text_pairs(
    n_samples=500, img_size=32, n_classes=5
)

# Show sample pairs
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
fig.suptitle('Sample Image-Text Pairs from Dataset', fontsize=14, fontweight='bold')
for i, ax in enumerate(axes.flat):
    img = images[i].permute(1, 2, 0).clamp(0, 1).numpy()
    ax.imshow(img)
    ax.set_title(texts[i], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f"Dataset: {len(images)} pairs, {len(set(labels))} classes")

## 2. Build CLIP + LoRA for Finetuning

### Learning Rate Schedule

Cosine annealing with warmup:

$$\eta_t = \begin{cases} \eta_{\max} \cdot t / T_{\text{warm}} & t < T_{\text{warm}} \\ \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\left(\pi \cdot \frac{t - T_{\text{warm}}}{T - T_{\text{warm}}}\right)\right) & t \geq T_{\text{warm}} \end{cases}$$

**Typical values:** $\eta_{\max} = 5 \times 10^{-4}$, warmup = 10% of steps, $\eta_{\min} = 0$.

### Gradient Accumulation

**Effective batch size** = micro_batch × accumulation_steps

For memory-constrained training, accumulate gradients over $K$ micro-batches before updating:

$$g_{\text{accumulated}} = \frac{1}{K}\sum_{k=1}^{K} g_k$$

**Example:** GPU fits batch=4, but you want effective batch=32 → accumulate over $K=8$ steps before updating.

In [ ]:
# LoRA layer (from notebook 01)
class LoRALinear(nn.Module):
    def __init__(self, original, rank=8, alpha=16):
        super().__init__()
        self.original = original
        for p in self.original.parameters():
            p.requires_grad = False
        in_f, out_f = original.in_features, original.out_features
        self.lora_A = nn.Parameter(torch.randn(in_f, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, out_f))
        self.scaling = alpha / rank

    def forward(self, x):
        return self.original(x) + (x @ self.lora_A @ self.lora_B) * self.scaling


# Small CLIP model
class SmallCLIP(nn.Module):
    def __init__(self, embed_dim=128, proj_dim=64, vocab_size=100):
        super().__init__()
        # Image encoder
        self.img_patch = nn.Conv2d(3, embed_dim, 4, 4)
        self.img_cls = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.img_pos = nn.Parameter(torch.randn(1, 65, embed_dim) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4,
                                          dim_feedforward=256, batch_first=True)
        self.img_enc = nn.TransformerEncoder(layer, num_layers=3)
        self.img_norm = nn.LayerNorm(embed_dim)
        self.img_proj = nn.Linear(embed_dim, proj_dim)

        # Text encoder
        self.tok_emb = nn.Embedding(vocab_size, embed_dim)
        self.txt_pos = nn.Embedding(32, embed_dim)
        layer2 = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4,
                                           dim_feedforward=256, batch_first=True)
        self.txt_enc = nn.TransformerEncoder(layer2, num_layers=3)
        self.txt_norm = nn.LayerNorm(embed_dim)
        self.txt_proj = nn.Linear(embed_dim, proj_dim)

        self.logit_scale = nn.Parameter(torch.ones(1) * np.log(1/0.07))

    def encode_image(self, x):
        B = x.shape[0]
        x = self.img_patch(x).flatten(2).transpose(1, 2)
        x = torch.cat([self.img_cls.expand(B,-1,-1), x], dim=1) + self.img_pos
        x = self.img_norm(self.img_enc(x)[:, 0])
        return F.normalize(self.img_proj(x), dim=-1)

    def encode_text(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B,-1)
        x = self.tok_emb(x) + self.txt_pos(pos)
        x = self.txt_norm(self.txt_enc(x)[:, 0])
        return F.normalize(self.txt_proj(x), dim=-1)

    def forward(self, images, text_ids):
        ie = self.encode_image(images)
        te = self.encode_text(text_ids)
        return self.logit_scale.exp() * ie @ te.T


# Tokenizer
all_words = set()
for t in texts:
    all_words.update(t.lower().split())
word2id = {w: i+2 for i, w in enumerate(sorted(all_words))}
word2id['[PAD]'] = 0
word2id['[CLS]'] = 1

def tokenize(text, max_len=16):
    ids = [1] + [word2id.get(w, 0) for w in text.lower().split()]
    ids = ids[:max_len] + [0] * max(0, max_len - len(ids))
    return torch.tensor(ids)

In [ ]:
# Step 1: "Pretrain" baseline (simulate a pretrained model)

class PairDataset(Dataset):
    def __init__(self, imgs, txts):
        self.imgs, self.txts = imgs, txts
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i): return self.imgs[i], tokenize(self.txts[i])

n_split = 400
train_ds = PairDataset(images[:n_split], texts[:n_split])
val_ds = PairDataset(images[n_split:], texts[n_split:])
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, drop_last=True)
val_dl = DataLoader(val_ds, batch_size=32)

model = SmallCLIP(embed_dim=128, proj_dim=64, vocab_size=len(word2id)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)

print("Step 1: Pretraining baseline...")
for epoch in range(15):
    model.train()
    for imgs, txts in train_dl:
        imgs, txts = imgs.to(device), txts.to(device)
        logits = model(imgs, txts)
        labels = torch.arange(len(imgs), device=device)
        loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2
        opt.zero_grad(); loss.backward(); opt.step()

print(f"  Pretrain done. Loss: {loss.item():.4f}")

# Save pretrained state
pretrained_state = {k: v.clone() for k, v in model.state_dict().items()}

In [ ]:
# Step 2: Apply LoRA and finetune

# Freeze everything
for p in model.parameters():
    p.requires_grad = False

# Replace projection layers with LoRA versions
model.img_proj = LoRALinear(model.img_proj, rank=4, alpha=8)
model.txt_proj = LoRALinear(model.txt_proj, rank=4, alpha=8)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total:,}")
print(f"Trainable (LoRA): {trainable:,} ({trainable/total*100:.2f}%)")

# Finetune with LoRA
lora_opt = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=5e-4
)

lora_losses = []
print("\nStep 2: Finetuning with LoRA...")
for epoch in range(20):
    model.train()
    epoch_loss = 0
    for imgs, txts in train_dl:
        imgs, txts = imgs.to(device), txts.to(device)
        logits = model(imgs, txts)
        labels = torch.arange(len(imgs), device=device)
        loss = (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2
        lora_opt.zero_grad(); loss.backward(); lora_opt.step()
        epoch_loss += loss.item()
    lora_losses.append(epoch_loss / len(train_dl))
    if (epoch+1) % 5 == 0:
        print(f"  Epoch {epoch+1}/20 | Loss: {lora_losses[-1]:.4f}")

# Plot finetuning loss
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lora_losses, linewidth=2, color='#E74C3C')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('LoRA Finetuning Loss', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Step 3: Evaluate — similarity matrix on validation set

model.eval()
with torch.no_grad():
    val_imgs = torch.stack(images[n_split:n_split+10]).to(device)
    val_txts = torch.stack([tokenize(t) for t in texts[n_split:n_split+10]]).to(device)
    img_emb = model.encode_image(val_imgs)
    txt_emb = model.encode_text(val_txts)

fig = plot_similarity_matrix(
    img_emb, txt_emb,
    labels_a=[f'img_{i}' for i in range(10)],
    labels_b=[t[:18] for t in texts[n_split:n_split+10]],
    title='After LoRA Finetuning: Image-Text Similarity'
)
plt.savefig('../assets/clip_finetuned_similarity.png', dpi=150, bbox_inches='tight')
plt.show()

### Evaluation Metrics for CLIP

- **Recall@K:** $R@K = \frac{|\{q : \text{correct match in top-K results}\}|}{|\text{queries}|}$. Common: R@1, R@5, R@10
- **Median Rank:** Median position of the correct match in the ranked list. Lower is better.
- **Mean Reciprocal Rank (MRR):** $\text{MRR} = \frac{1}{N}\sum_{i=1}^{N}\frac{1}{\text{rank}_i}$

| Stage | R@1 (typical) |
|-------|---------------|
| Before LoRA FT | 30–40% |
| After LoRA FT | 70–85% on domain data |

In [ ]:
# Step 4: Save LoRA adapter (only the trainable weights!)

lora_state = {k: v for k, v in model.state_dict().items() if 'lora' in k}
torch.save(lora_state, '../assets/clip_lora_adapter.pt')

full_size = sum(v.numel() * v.element_size() for v in model.state_dict().values())
lora_size = sum(v.numel() * v.element_size() for v in lora_state.values())

print(f"Full model size:  {full_size/1024:.1f} KB")
print(f"LoRA adapter:     {lora_size/1024:.1f} KB")
print(f"Savings:          {(1-lora_size/full_size)*100:.1f}%")
print(f"\nSaved to: ../assets/clip_lora_adapter.pt")
print("You can share just this tiny file to share your finetuned model!")

## Using Pretrained CLIP + PEFT (Real-World Template)

```python
import open_clip
from peft import LoraConfig, get_peft_model

# Load pretrained CLIP
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='laion2b_s34b_b79k'
)

# Freeze, then add LoRA
for p in model.parameters():
    p.requires_grad = False

config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=['out_proj', 'mlp.c_fc', 'mlp.c_proj'],
    lora_dropout=0.05,
)
model = get_peft_model(model, config)
model.print_trainable_parameters()
# -> trainable: 0.5% of total!
```

---
**Next:** Module 05 - Advanced Topics (LLaVA, Audio+Vision, Deployment)